In [1]:
from src.vasijas.reporting import ProcessingReport
from _bootstrap import setup
import pandas as pd
import shutil as sh

PATHS = setup()
report = ProcessingReport()

# Archivos
archivo_ubicacion: dict[str, str] = {p.stem: p.parent.name for p in PATHS.DATA_CORREGIDOS.rglob("*.png")}
# Preview
dict(list(archivo_ubicacion.items())[:10])

{'pn_74_64': 'Puerta Norte',
 'pn_9_2': 'Puerta Norte',
 'pn_25_2': 'Puerta Norte',
 'pn_27_5': 'Puerta Norte',
 'pn_26_1': 'Puerta Norte',
 'pn_24_4': 'Puerta Norte',
 'pn_60_37': 'Puerta Norte',
 'pn_24_2': 'Puerta Norte',
 'pn_76_69': 'Puerta Norte',
 'pn_26_2': 'Puerta Norte'}

In [2]:
# Leer CSV
archivo_csv = PATHS.DATA_CORREGIDOS / "metadata.csv"
df = pd.read_csv(archivo_csv)

df.head()

,Identificador,Tiene asa,Tendencia Formal,Borde,Cuello,Cuerpo,Base,Clase Ana,Diametro Borde,Diametro Base,Altura Total,Cronologia
0,al_03a,No,II,1,B,2,2,11.0,182,54,53,ibero
1,al_03b,No,II,1,B,2,2,11.0,178,57,46,ibero
2,al_03c,No,II,1,B,2,2,11.0,185,56,49,ibero
3,al_03d,No,II,1,B,2,2,11.0,160,41,45,ibero
4,al_03e,No,II,1,B,2,2,11.0,164,54,44,ibero


In [3]:
# Agregar columna ubicacion a partir del diccionario archivo_ubicacion
# Busca el Identificador en el diccionario y carga la ubicación (nombre de carpeta padre)
df["Ubicacion"] = df["Identificador"].astype(str).map(archivo_ubicacion)

# Verificar valores no encontrados
no_encontrados = df[df["Ubicacion"].isna()]["Identificador"].tolist()
if no_encontrados:
    mensaje = f"{len(no_encontrados)} identificadores sin ubicación: {no_encontrados}"
else:
    mensaje =  "Todos los identificadores tienen ubicación asignada."

print (mensaje)

Todos los identificadores tienen ubicación asignada.


In [4]:
# Copiar toda la estructura de DATA_CORREGIDOS a DATA_UBICADOS
for archivo in PATHS.DATA_CORREGIDOS.rglob("*"):
    if archivo.is_file():
        # Calcular ruta relativa respecto al origen
        ruta_relativa = archivo.relative_to(PATHS.DATA_CORREGIDOS)
        destino = PATHS.DATA_UBICADOS / ruta_relativa

        # Crear directorio destino si no existe
        destino.parent.mkdir(parents=True, exist_ok=True)

        # Copiar archivo
        sh.copy2(archivo, destino)

print(f"Archivos copiados desde {PATHS.DATA_CORREGIDOS} hacia {PATHS.DATA_UBICADOS}")

Archivos copiados desde /home/rafael/Dev/vasijas/data/02.corregidos hacia /home/rafael/Dev/vasijas/data/03.ubicados


In [5]:
# Guardar nuevo archivo metadata

archivo_salida = PATHS.DATA_UBICADOS / "metadata.csv"
df.to_csv(archivo_salida, index=False)

print("Archivo generado:", archivo_salida)

Archivo generado: /home/rafael/Dev/vasijas/data/03.ubicados/metadata.csv
